In [1]:
# GPU Check
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU! Go to: Runtime > Change runtime type > T4 GPU')

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# Install dependencies
!pip install transformers peft bitsandbytes accelerate -q
print('Core packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
Core packages installed!


In [3]:
#Setup output folders
import os

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_PATH = '/content/adapters'

OUTPUT_DIR = '/content/quantized'
MERGED_DIR = f'{OUTPUT_DIR}/merged_fp16'
INT8_DIR   = f'{OUTPUT_DIR}/model-int8'
INT4_DIR   = f'{OUTPUT_DIR}/model-int4'
GGUF_PATH  = f'{OUTPUT_DIR}/model.gguf'

for d in [OUTPUT_DIR, MERGED_DIR, INT8_DIR, INT4_DIR]:
    os.makedirs(d, exist_ok=True)

print('Output folders created:')
print(f'   {OUTPUT_DIR}/')
print(f'   ├── merged_fp16/')
print(f'   ├── model-int8/')
print(f'   ├── model-int4/')
print(f'   └── model.gguf')

Output folders created:
   /content/quantized/
   ├── merged_fp16/
   ├── model-int8/
   ├── model-int4/
   └── model.gguf


In [ ]:
#Load FP16 model & measure baseline
from transformers import AutoModelForCausalLM, AutoTokenizer
import time, gc

def get_vram_gb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9
    return 0.0

def get_dir_size_gb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1e9

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Loading tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print('Loading FP16 model...')
clear_gpu()
vram_before = get_vram_gb()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)

# Load LoRA adapter
if ADAPTER_PATH and os.path.exists(ADAPTER_PATH):
    from peft import PeftModel
    print(f'Loading LoRA adapter from {ADAPTER_PATH}...')
    model_fp16 = PeftModel.from_pretrained(model_fp16, ADAPTER_PATH)
    print('Merging adapter into base model...')
    model_fp16 = model_fp16.merge_and_unload()  # Bake adapter into weights
    print('Adapter merged!')
else:
    print('[INFO] No adapter found. Using base model (add adapter path for full pipeline)')

vram_fp16 = get_vram_gb() - vram_before
print(f'\nFP16 model loaded!')
print(f'   VRAM used: {vram_fp16:.2f} GB')
print(f'   Params: {sum(p.numel() for p in model_fp16.parameters()):,}')

Loading tokenizer: TinyLlama/TinyLlama-1.1B-Chat-v1.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading LoRA adapter from /content/adapters...
Merging adapter into base model...
Adapter merged!

FP16 model loaded!
   VRAM used: 2.21 GB
   Params: 1,100,048,384


In [5]:
#Measure FP16 inference speed
TEST_PROMPT = '### Instruction:\nWhat is compound interest?\n\n### Response:\n'

def measure_speed(model, tokenizer, prompt, n_tokens=50, label=''):
    inputs = tokenizer(prompt, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    # Warmup
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)

    # Actual timing
    if torch.cuda.is_available(): torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=n_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    new_tokens = out.shape[1] - inputs['input_ids'].shape[1]
    tps = new_tokens / elapsed
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    print(f'\n[{label}]')
    print(f'  Speed:    {tps:.1f} tokens/sec')
    print(f'  Time:     {elapsed:.2f}s for {new_tokens} tokens')
    print(f'  Response: {response[:150]}...')
    return round(tps, 1)

fp16_tps = measure_speed(model_fp16, tokenizer, TEST_PROMPT, label='FP16')

# Save FP16 model
print(f'\nSaving FP16 merged model to {MERGED_DIR}...')
model_fp16.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
fp16_size = get_dir_size_gb(MERGED_DIR)
print(f'Saved! Size: {fp16_size:.2f} GB')

# Free memory
del model_fp16
clear_gpu()
print('GPU memory freed')


[FP16]
  Speed:    12.7 tokens/sec
  Time:     3.95s for 50 tokens
  Response: Compound interest is a calculation that adds together simple interest over time. Example: If you deposit $1000 and earn 10% per year, the total amount...

Saving FP16 merged model to /content/quantized/merged_fp16...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved! Size: 2.20 GB
GPU memory freed


In [ ]:
from transformers import BitsAndBytesConfig

print('Loading model in INT8...')
print('(2x smaller than FP16, almost same quality)')

clear_gpu()
vram_before = get_vram_gb()

bnb_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,  # Outlier threshold: weights > 6.0 stay in FP16
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,              # Load from our saved FP16 merged model
    quantization_config=bnb_8bit,
    device_map='auto',
)

vram_int8 = get_vram_gb() - vram_before
print(f'\nINT8 model loaded!')
print(f'   VRAM used: {vram_int8:.2f} GB')
print(f'   vs FP16:   {vram_fp16:.2f} GB')
if vram_fp16 > 0:
    print(f'   Savings:   {((vram_fp16 - vram_int8)/vram_fp16*100):.0f}% less VRAM')
else:
    print('   Savings:   Cannot calculate VRAM savings without GPU data (vram_fp16 is 0)')

int8_tps = measure_speed(model_int8, tokenizer, TEST_PROMPT, label='INT8')

# Save INT8 model and info
print(f'\nSaving INT8 model and config to {INT8_DIR}...')
model_int8.save_pretrained(INT8_DIR) # Save the model itself
tokenizer.save_pretrained(INT8_DIR)
int8_disk_size = get_dir_size_gb(INT8_DIR) # Measure disk size after saving
print(f'INT8 model saved! Disk size: {int8_disk_size:.2f} GB')

with open(f'{INT8_DIR}/README.md', 'w') as f:
    f.write(f'# INT8 Quantized Model\n\n')
    f.write(f'Base: {BASE_MODEL}\n')
    f.write(f'VRAM: ~{vram_int8:.2f} GB (In-memory)\n')
    f.write(f'Disk Size: ~{int8_disk_size:.2f} GB (Weights saved mostly as FP16 with quantization config)\n')
    f.write(f'Speed: {int8_tps} tok/s\n\n')
    f.write('## Load code:\n```python\n')
    f.write('from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n')
    f.write('bnb = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)\n')
    f.write(f'model = AutoModelForCausalLM.from_pretrained("'+INT8_DIR+'", quantization_config=bnb)\n```') # Load from saved path
print(f'INT8 config saved to {INT8_DIR}')

del model_int8
clear_gpu()
print('GPU memory freed')


Loading model in INT8...
(2x smaller than FP16, almost same quality)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


INT8 model loaded!
   VRAM used: 1.24 GB
   vs FP16:   2.21 GB
   Savings:   44% less VRAM

[INT8]
  Speed:    4.9 tokens/sec
  Time:     10.29s for 50 tokens
  Response: Compound interest is a calculation that adds together simple interest over time. Example: If you deposit $100 and earn 10% each year, the total amount...

Saving INT8 model and config to /content/quantized/model-int8...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 model saved! Disk size: 1.24 GB
INT8 config saved to /content/quantized/model-int8
GPU memory freed


In [ ]:
# INT4 / NF4 Quantization
print('Loading model in INT4 (NF4)...')
print('(4x smaller than FP16!)')

clear_gpu()
vram_before = get_vram_gb()

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',           
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,       
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    quantization_config=bnb_4bit,
    device_map='auto',
)

vram_int4 = get_vram_gb() - vram_before
print(f'\nINT4 model loaded!')
print(f'   VRAM used: {vram_int4:.2f} GB')
print(f'   vs FP16:   {vram_fp16:.2f} GB')
if vram_fp16 > 0:
    print(f'   Savings:   {((vram_fp16 - vram_int4)/vram_fp16*100):.0f}% less VRAM')
else:
    print('   Savings:   Cannot calculate VRAM savings without GPU data (vram_fp16 is 0)')

int4_tps = measure_speed(model_int4, tokenizer, TEST_PROMPT, label='INT4 NF4')

print(f'\nSaving INT4 model and config to {INT4_DIR}...')
model_int4.save_pretrained(INT4_DIR)
tokenizer.save_pretrained(INT4_DIR)
int4_disk_size = get_dir_size_gb(INT4_DIR)
print(f'INT4 model saved! Disk size: {int4_disk_size:.2f} GB')

with open(f'{INT4_DIR}/README.md', 'w') as f:
    f.write(f'# INT4 (NF4) Quantized Model\n\n')
    f.write(f'Base: {BASE_MODEL}\n')
    f.write(f'VRAM: ~{vram_int4:.2f} GB (In-memory)\n')
    f.write(f'Disk Size: ~{int4_disk_size:.2f} GB (Weights saved mostly as FP16 with quantization config)\n')
    f.write(f'Speed: {int4_tps} tok/s\n\n')
    f.write('## Load code:\n```python\n')
    f.write('from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n')
    f.write('bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n')
    f.write('                         bnb_4bit_compute_dtype=torch.float16)\n')
    f.write(f'model = AutoModelForCausalLM.from_pretrained("'+INT4_DIR+'", quantization_config=bnb)\n```') # Load from saved path
print(f' INT4 config saved to {INT4_DIR}')

del model_int4
clear_gpu()
print('GPU memory freed ')


Loading model in INT4 (NF4)...
(4x smaller than FP16!)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


INT4 model loaded!
   VRAM used: 0.77 GB
   vs FP16:   2.21 GB
   Savings:   65% less VRAM

[INT4 NF4]
  Speed:    18.2 tokens/sec
  Time:     2.52s for 46 tokens
  Response: Compound interest is a calculation that adds together simple interest. This concept is important in finance and helps in decision making.

### Input:
...

Saving INT4 model and config to /content/quantized/model-int4...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 model saved! Disk size: 0.77 GB
 INT4 config saved to /content/quantized/model-int4
GPU memory freed ✅


In [ ]:
#  Install llama.cpp
import subprocess, sys

LLAMA_CPP_DIR = '/content/llama.cpp'

if not os.path.exists(LLAMA_CPP_DIR):
    print('Cloning llama.cpp...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/ggerganov/llama.cpp', LLAMA_CPP_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('llama.cpp cloned!')
    else:
        print(f'Clone failed: {result.stderr}')
else:
    print('llama.cpp already present')

req = f'{LLAMA_CPP_DIR}/requirements.txt'
if os.path.exists(req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req, '-q'])
    print('llama.cpp requirements installed')

Cloning llama.cpp...
llama.cpp cloned!
llama.cpp requirements installed


In [9]:
print('Upgrading tokenizers package to fix potential backend issues...')
!pip install --upgrade tokenizers -q
print('tokenizers upgraded!')


Upgrading tokenizers package to fix potential backend issues...
tokenizers upgraded!


In [10]:
print('Reinstalling specific versions of transformers and tokenizers for compatibility...')
!pip install transformers==4.32.1 tokenizers==0.13.3 -q
print('transformers and tokenizers reinstalled!')

Reinstalling specific versions of transformers and tokenizers for compatibility...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 33.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 128.7 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
transformers and tokenizers reinstalled!


In [11]:
import sys
import os

print('Python sys.path:')
for p in sys.path:
    print(p)

print('\nTrying to import LlamaConfig explicitly...')
try:
    from transformers import LlamaConfig
    print('LlamaConfig imported successfully from transformers!')
except ImportError as e:
    print(f'Failed to import LlamaConfig: {e}')
    print('This indicates a problem with the transformers installation or environment path.')


Python sys.path:
/content
/env/python
/usr/lib/python312.zip
/usr/lib/python3.12
/usr/lib/python3.12/lib-dynload

/usr/local/lib/python3.12/dist-packages
/usr/lib/python3/dist-packages
/usr/local/lib/python3.12/dist-packages/IPython/extensions
/root/.ipython

Trying to import LlamaConfig explicitly...
LlamaConfig imported successfully from transformers!


In [ ]:
TEMP_GGUF = f'{OUTPUT_DIR}/model_fp16.gguf'
QUANT_TYPE = 'q4_0' 
convert_scripts = [
    f'{LLAMA_CPP_DIR}/convert_hf_to_gguf.py',
    f'{LLAMA_CPP_DIR}/convert.py',
]
convert_script = None
for s in convert_scripts:
    if os.path.exists(s):
        convert_script = s
        break

if not convert_script:
    print(' Conversion script not found in llama.cpp')
    print('   Try: !ls /content/llama.cpp/*.py')
else:
    print(f'Using conversion script: {convert_script}')
    print(f'Converting {MERGED_DIR} → {TEMP_GGUF}...')
    print('(This takes 2-5 minutes)')

    from huggingface_hub import snapshot_download
    import shutil

    print('Ensuring tokenizer.model is available for GGUF conversion...')
    model_cache_path = snapshot_download(repo_id=BASE_MODEL)
    src_tokenizer_model_path = os.path.join(model_cache_path, 'tokenizer.model')

    dst_tokenizer_model_path = os.path.join(MERGED_DIR, 'tokenizer.model')

    if os.path.exists(src_tokenizer_model_path):
        if not os.path.exists(dst_tokenizer_model_path) or os.path.getsize(src_tokenizer_model_path) != os.path.getsize(dst_tokenizer_model_path):
            print(f'Copying tokenizer.model from {src_tokenizer_model_path} to {dst_tokenizer_model_path}')
            shutil.copy(src_tokenizer_model_path, dst_tokenizer_model_path)
            print('tokenizer.model copied!')
        else:
            print('tokenizer.model already exists and is up-to-date in MERGED_DIR.')
    else:
        print(f'tokenizer.model not found at {src_tokenizer_model_path}. GGUF conversion might fail.')
    current_env = os.environ.copy()

    try:
        result = subprocess.run(
            [sys.executable, convert_script,
             MERGED_DIR,
             '--outfile', TEMP_GGUF,
             '--outtype', 'f16',
             '--verbose'],
            timeout=600, check=True,
            env=current_env
        )

        if result.returncode == 0:
            size = os.path.getsize(TEMP_GGUF) / 1e9
            print(f'FP16 GGUF created! Size: {size:.2f} GB')
    except subprocess.TimeoutExpired as e:
        print(f'Conversion timed out after {e.timeout} seconds!')
        print('\nManual command:')
        print(f'!python {convert_script} {MERGED_DIR} --outfile {TEMP_GGUF} --outtype f16 --verbose')
    except subprocess.CalledProcessError as e:
        print(f'Conversion failed with CalledProcessError!')
        print('Return code:', e.returncode)
        print('\nManual command:')
        print(f'!python {convert_script} {MERGED_DIR} --outfile {TEMP_GGUF} --outtype f16 --verbose')
    except Exception as e:
        print(f'An unexpected error occurred during conversion: {e}')

Using conversion script: /content/llama.cpp/convert_hf_to_gguf.py
Converting /content/quantized/merged_fp16 → /content/quantized/model_fp16.gguf...
(This takes 2-5 minutes)
Ensuring tokenizer.model is available for GGUF conversion...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Copying tokenizer.model from /root/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer.model to /content/quantized/merged_fp16/tokenizer.model
tokenizer.model copied!
FP16 GGUF created! Size: 2.20 GB


In [13]:
# Running the GGUF conversion command directly to get full error logs
!python /content/llama.cpp/convert_hf_to_gguf.py /content/quantized/merged_fp16 --outfile /content/quantized/model_fp16.gguf --outtype f16 --verbose

INFO:hf-to-gguf:Loading model: merged_fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.wei

In [ ]:
print('Building llama.cpp (for quantize binary)...')

build_result_make = subprocess.run(
    ['make', '-j', '4', 'llama-quantize'],
    cwd=LLAMA_CPP_DIR,
    capture_output=True, text=True
)

quantize_bin = f'{LLAMA_CPP_DIR}/llama-quantize'

if build_result_make.returncode == 0 and os.path.exists(quantize_bin):
    print('llama-quantize built with make!')
else:
    print('make build failed.')
    print('STDOUT:', build_result_make.stdout[-1000:] if build_result_make.stdout else 'No stdout')
    print('STDERR:', build_result_make.stderr[-1000:] if build_result_make.stderr else 'No stderr')
    print('Trying alternative: cmake')

    # Alternative: cmake build
    cmake_dir = f'{LLAMA_CPP_DIR}/build'
    os.makedirs(cmake_dir, exist_ok=True)

    print('Running cmake configuration...')
    cmake_config_result = subprocess.run(
        ['cmake', '..', '-DLLAMA_CUBLAS=OFF'],
        cwd=cmake_dir,
        capture_output=True, text=True, check=False
    )
    print('STDOUT:', cmake_config_result.stdout[-1000:] if cmake_config_result.stdout else 'No stdout')
    print('STDERR:', cmake_config_result.stderr[-1000:] if cmake_config_result.stderr else 'No stderr')

    if cmake_config_result.returncode != 0:
        print(' CMake configuration failed. Cannot proceed with build.')
    else:
        print('Running cmake build...')
        cmake_build_result = subprocess.run(
            ['cmake', '--build', '.', '--config', 'Release', '-j', '4'],
            cwd=cmake_dir,
            capture_output=True, text=True, check=False 
        )
        print('STDOUT:', cmake_build_result.stdout[-1000:] if cmake_build_result.stdout else 'No stdout')
        print('STDERR:', cmake_build_result.stderr[-1000:] if cmake_build_result.stderr else 'No stderr')

        if cmake_build_result.returncode == 0:
            quantize_bin_alt = f'{cmake_dir}/bin/llama-quantize'
            if os.path.exists(quantize_bin_alt):
                print(' Built with cmake!')
                quantize_bin = quantize_bin_alt
            else:
                print(' CMake build succeeded, but llama-quantize binary not found.')
                print('[FALLBACK] Using FP16 GGUF without further quantization')
                
                if os.path.exists(TEMP_GGUF):
                    shutil.move(TEMP_GGUF, GGUF_PATH)
                else:
                    print(f'Warning: {TEMP_GGUF} not found for fallback.')
                gguf_size = os.path.getsize(GGUF_PATH) / 1e9 if os.path.exists(GGUF_PATH) else 0
                print(f'GGUF size: {gguf_size:.2f} GB')
        else:
            print('CMake build failed.')
            print('[FALLBACK] Using FP16 GGUF without further quantization')
            
            if os.path.exists(TEMP_GGUF):
                shutil.move(TEMP_GGUF, GGUF_PATH)
            else:
                print(f'Warning: {TEMP_GGUF} not found for fallback.')
            gguf_size = os.path.getsize(GGUF_PATH) / 1e9 if os.path.exists(GGUF_PATH) else 0
            print(f'GGUF size: {gguf_size:.2f} GB')


if os.path.exists(quantize_bin) and quantize_bin != f'{LLAMA_CPP_DIR}/llama-quantize': 
    if os.path.exists(TEMP_GGUF):
        print(f'Quantizing {TEMP_GGUF} → {GGUF_PATH} ({QUANT_TYPE})...')
        q_result = subprocess.run(
            [quantize_bin, TEMP_GGUF, GGUF_PATH, QUANT_TYPE],
            capture_output=True, text=True, timeout=300, check=True
        )
        if q_result.returncode == 0:
            os.remove(TEMP_GGUF)  
            gguf_size = os.path.getsize(GGUF_PATH) / 1e9
            print(f' GGUF quantized! Size: {gguf_size:.2f} GB ({QUANT_TYPE})')
        else:
            print('Quantize failed:', q_result.stderr[-500:])
            import shutil
            shutil.move(TEMP_GGUF, GGUF_PATH)
            gguf_size = os.path.getsize(GGUF_PATH) / 1e9
            print(f'Using FP16 GGUF instead: {gguf_size:.2f} GB')
    else:
        print(f'Warning: {TEMP_GGUF} not found for quantization. Skipping quantization step.')
        if not os.path.exists(GGUF_PATH): 
            print('No GGUF file available after build attempts.')
else:
    print('Quantization binary not successfully built or found. Skipping quantization.')
    if os.path.exists(TEMP_GGUF) and not os.path.exists(GGUF_PATH): 
        shutil.move(TEMP_GGUF, GGUF_PATH)
        gguf_size = os.path.getsize(GGUF_PATH) / 1e9
        print(f'Using FP16 GGUF as final output: {gguf_size:.2f} GB')
    elif os.path.exists(GGUF_PATH):
        gguf_size = os.path.getsize(GGUF_PATH) / 1e9
        print(f'GGUF size already established: {gguf_size:.2f} GB')
    else:
        print('No GGUF file available after build attempts or fallback.')



Building llama.cpp (for quantize binary)...
make build failed.
STDOUT: No stdout
STDERR: Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.

Trying alternative: cmake
Running cmake configuration...
STDOUT: U
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.11
-- ggml commit:  5e9c635
-- Found OpenSSL: /usr/lib/x8

In [ ]:
import pandas as pd
if 'int8_disk_size' not in locals():
    int8_disk_size = get_dir_size_gb(INT8_DIR) if os.path.exists(INT8_DIR) else 0.0
if 'int4_disk_size' not in locals():
    int4_disk_size = get_dir_size_gb(INT4_DIR) if os.path.exists(INT4_DIR) else 0.0

summary_data = {
    'Format': ['FP16 (Baseline)', 'INT8', 'INT4 (NF4)', f'GGUF ({QUANT_TYPE})'],
    'Size (Disk, GB)': [
        f'{fp16_size:.2f} (Model only)',
        f'{int8_disk_size:.2f} (FP16+Config)',
        f'{int4_disk_size:.2f} (FP16+Config)', 
        f'{gguf_size:.2f} (CPU format)'
    ],
    'VRAM (Load, GB)': [
        f'{vram_fp16:.2f}',
        f'{vram_int8:.2f}',
        f'{vram_int4:.2f}',
        'N/A (CPU only)'
    ],
    'Speed (Tokens/sec)': [
        f'{fp16_tps}',
        f'{int8_tps}',
        f'{int4_tps}',
        'Not measured (CPU)'
    ],
    'Quality (vs FP16)': [
        'Baseline',
        'Good (Minor drop)',
        'Acceptable (Noticeable drop)',
        'Good (Similar to INT4)'
    ]
}

summary_df = pd.DataFrame(summary_data)

print('\nQuantization Summary:')
display(summary_df)

if vram_fp16 == 0.0:
    print('\nNote: VRAM (Load, GB) values are 0.00 because no GPU was detected. Expected VRAM usage for TinyLlama 1.1B on GPU would be approximately:')
    print('  FP16: ~2.2 GB')
    print('  INT8: ~1.1 GB')
    print('  INT4: ~0.6 GB')



Quantization Summary:


,Format,"Size (Disk, GB)","VRAM (Load, GB)",Speed (Tokens/sec),Quality (vs FP16)
0,FP16 (Baseline),2.20 (Model only),2.21,12.7,Baseline
1,INT8,1.24 (FP16+Config),1.24,4.9,Good (Minor drop)
2,INT4 (NF4),0.77 (FP16+Config),0.77,18.2,Acceptable (Noticeable drop)
3,GGUF (q4_0),0.64 (CPU format),N/A (CPU only),Not measured (CPU),Good (Similar to INT4)
